# Bronze (Spark Declarative Pipeline)

Reads the raw files we landed in the Volume and turns them into Delta tables.
Bronze keeps the data as-is (all strings, no trimming/casting) - that happens in Silver.

We add non-null key expectations here to flag bad rows on the way in.
Bronze keeps everything (warn only) - Silver is where we actually drop/clean.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.types import StructType, StructField, StringType

# Pipeline config (set these in the pipeline settings / DAB; defaults for dev)
catalog = spark.conf.get("catalog", "rearc_dev")
raw_schema = spark.conf.get("raw_schema", "dataquest_raw")
bronze_schema = spark.conf.get("bronze_schema", "dataquest_bronze")
volume = spark.conf.get("volume", "landing")

bls_path = f"/Volumes/{catalog}/{raw_schema}/{volume}/bls/pr"
pop_path = f"/Volumes/{catalog}/{raw_schema}/{volume}/population"

## BLS data file (the facts)

In [0]:
# Explicit schema so we enforce the expected columns (everything stays string in bronze)
pr_data_schema = StructType([
    StructField("series_id", StringType()),
    StructField("year", StringType()),
    StructField("period", StringType()),
    StructField("value", StringType()),
    StructField("footnote_codes", StringType()),
])


@dp.materialized_view(name=f"{bronze_schema}.bronze_pr_data")
@dp.expect_all({
    "valid_series_id": "series_id IS NOT NULL",
    "valid_year": "year IS NOT NULL",
    "valid_period": "period IS NOT NULL",
})
def bronze_pr_data():
    return (spark.read
            .option("sep", "\t")
            .option("header", True)
            .schema(pr_data_schema)
            .csv(f"{bls_path}/pr.data.1.AllData"))

## BLS series file (the series dimension)

In [0]:
pr_series_schema = StructType([
    StructField("series_id", StringType()),
    StructField("sector_code", StringType()),
    StructField("class_code", StringType()),
    StructField("measure_code", StringType()),
    StructField("duration_code", StringType()),
    StructField("seasonal", StringType()),
    StructField("base_year", StringType()),
    StructField("footnote_codes", StringType()),
    StructField("begin_year", StringType()),
    StructField("begin_period", StringType()),
    StructField("end_year", StringType()),
    StructField("end_period", StringType()),
])


@dp.materialized_view(name=f"{bronze_schema}.bronze_pr_series")
@dp.expect_all({"valid_series_id": "series_id IS NOT NULL"})
def bronze_pr_series():
    return (spark.read
            .option("sep", "\t")
            .option("header", True)
            .schema(pr_series_schema)
            .csv(f"{bls_path}/pr.series"))

## BLS mapping / lookup files
Small code->text tables. Same tab-separated format, so we load them in a loop.

In [0]:
mapping_files = {
    "bronze_pr_measure": "pr.measure",
    "bronze_pr_sector": "pr.sector",
    "bronze_pr_class": "pr.class",
    "bronze_pr_duration": "pr.duration",
    "bronze_pr_seasonal": "pr.seasonal",
    "bronze_pr_period": "pr.period",
    "bronze_pr_footnote": "pr.footnote",
}


def define_mapping_table(table_name, file_name):
    @dp.materialized_view(name=f"{bronze_schema}.{table_name}")
    def mapping_table():
        return (spark.read
                .option("sep", "\t")
                .option("header", True)
                .csv(f"{bls_path}/{file_name}"))
    return mapping_table


for tbl, fname in mapping_files.items():
    define_mapping_table(tbl, fname)

## Population API (JSON)
The API returns one object with a "data" array, so we explode it into rows.

In [0]:
@dp.materialized_view(
    name=f"{bronze_schema}.bronze_population",
    table_properties={"delta.columnMapping.mode": "name"}
)
@dp.expect_all({
    "valid_year": "Year IS NOT NULL",
    "valid_population": "Population IS NOT NULL",
})
def bronze_population():
    raw = spark.read.option("multiline", "true").json(f"{pop_path}/population.json")
    return raw.selectExpr("explode(data) as r").select("r.*")